In [1]:
# 데이터 불러오기
import pandas as pd

file_path = "/content/drive/MyDrive/JeonseGuard/실거래가/매매/연립다세대/202501_연립다세대_매매_실거래가.csv" # CSV 파일 경로 지정
df = pd.read_csv(file_path, encoding='cp949') # CP949 인코딩

In [2]:
# 상위 5개 확인
df.head()

,NO,시군구,번지,본번,부번,건물명,전용면적(㎡),대지권면적(㎡),계약년월,계약일,...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
0,1,서울특별시 강남구 개포동,1255-10,1255,10,베네스트,71.4500,39.5200,202501,31,...,5,개인,개인,2003,개포로25길 13-10,20250225,중개거래,서울 강남구,-,다세대
1,2,서울특별시 양천구 목동,797-7,797,7,연우NEXT,29.6600,17.0000,202501,31,...,6,개인,개인,2015,목동중앙서로 28,-,직거래,-,25.02.18,다세대
2,3,경상북도 영덕군 영덕읍 덕곡리,216-10,216,10,서희예다움,134.7622,112.4041,202501,31,...,4,개인,개인,2014,강변길 38,-,직거래,-,25.02.19,연립
3,4,서울특별시 강서구 화곡동,424-85,424,85,비에스하우징,33.2700,20.0600,202501,31,...,4,개인,개인,2018,곰달래로19가길 30-5,20250221,중개거래,서울 금천구,-,다세대
4,5,인천광역시 남동구 만수동,111-471,111,471,롯데맨션(111-471)D동,29.5200,8.8000,202501,31,...,2,개인,개인,1996,서판로63번길 4-4,-,직거래,-,25.02.17,다세대


In [3]:
# 컬럼명 확인
print(df.columns)

Index(['NO', '시군구', '번지', '본번', '부번', '건물명', '전용면적(㎡)', '대지권면적(㎡)', '계약년월',
       '계약일', '거래금액(만원)', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형',
       '중개사소재지', '등기일자', '주택유형'],
      dtype='object')


In [4]:
# 특정 컬럼 값만 확인
df["거래금액(만원)"].head()

,거래금액(만원)
0,"71,500"
1,"21,500"
2,"25,000"
3,"24,000"
4,"5,000"


In [5]:
# 변경 매핑 딕셔너리 정의
renamed_columns = {
    "시군구": "address",
    "본번": "bun",
    "부번": "ji",
    "층": "floor",
    "전용면적(㎡)": "area",
    "계약년월": "contract_year_month",
    "거래금액(만원)": "price",
    "주택유형": "housing_type"
}

In [6]:
# 매핑에 해당하는 컬럼만 필터링
df = df.rename(columns=renamed_columns)
sale_df = df[list(renamed_columns.values())].copy()
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,서울특별시 강남구 개포동,1255,10,5,71.4500,202501,"71,500",다세대
1,서울특별시 양천구 목동,797,7,6,29.6600,202501,"21,500",다세대
2,경상북도 영덕군 영덕읍 덕곡리,216,10,4,134.7622,202501,"25,000",연립
3,서울특별시 강서구 화곡동,424,85,4,33.2700,202501,"24,000",다세대
4,인천광역시 남동구 만수동,111,471,2,29.5200,202501,"5,000",다세대


In [7]:
# 쉼표 제거 및 문자열을 정수로 변환
sale_df["price"] = (
    sale_df["price"]
    .astype(str) # 문자열로 변환 (안전)
    .str.replace(",", "") # 쉼표 제거
    .astype(int) # 정수형으로 변환
    * 10000 # 만원 → 원 변환
)

In [8]:
# 변환된 price 컬럼 확인
sale_df[["price"]].head()

,price
0,715000000
1,215000000
2,250000000
3,240000000
4,50000000


In [9]:
# price 컬럼의 값을 쉼표가 포함된 문자열로 변환
sale_df["price"] = sale_df["price"].apply(lambda x: format(x, ","))
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,서울특별시 강남구 개포동,1255,10,5,71.4500,202501,"715,000,000",다세대
1,서울특별시 양천구 목동,797,7,6,29.6600,202501,"215,000,000",다세대
2,경상북도 영덕군 영덕읍 덕곡리,216,10,4,134.7622,202501,"250,000,000",연립
3,서울특별시 강서구 화곡동,424,85,4,33.2700,202501,"240,000,000",다세대
4,인천광역시 남동구 만수동,111,471,2,29.5200,202501,"50,000,000",다세대


In [10]:
# 결측치 확인
missing_counts = sale_df.isnull().sum()
print("📌 결측치가 있는 컬럼: ", missing_counts[missing_counts > 0])

📌 결측치가 있는 컬럼:  Series([], dtype: int64)


In [11]:
# 전체 중복된 행의 수 확인
duplicate_count = sale_df.duplicated().sum()
print(f"📌 중복된 행의 수: {duplicate_count}")

📌 중복된 행의 수: 189


In [12]:
# 중복 제거 (기본: 모든 열 기준, keep='first')
sale_df = sale_df.drop_duplicates().reset_index(drop=True)

In [13]:
# INSERT 구문 생성
insert_header = """INSERT INTO transaction_sale_rowhouse (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES """

In [14]:
# 행별로 SQL 값 문자열 생성
values = []

In [15]:
# 각 행을 values 리스트에 추가
for _, row in sale_df.iterrows():
    values.append(f"('{row['address']}', '{row['bun']}', '{row['ji']}', '{row['floor']}', '{row['area']}', '{row['contract_year_month']}', '{row['price']}', '{row['housing_type']}', NOW(), NOW())")

In [16]:
# INSERT 구문 상위 5개만 출력
preview_sql = insert_header + ",\n       ".join(values[:5]) + ";"
print(preview_sql)

INSERT INTO transaction_sale_rowhouse (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES ('서울특별시 강남구 개포동', '1255', '10', '5', '71.45', '202501', '715,000,000', '다세대', NOW(), NOW()),
       ('서울특별시 양천구 목동', '797', '7', '6', '29.66', '202501', '215,000,000', '다세대', NOW(), NOW()),
       ('경상북도 영덕군 영덕읍 덕곡리', '216', '10', '4', '134.7622', '202501', '250,000,000', '연립', NOW(), NOW()),
       ('서울특별시 강서구 화곡동', '424', '85', '4', '33.27', '202501', '240,000,000', '다세대', NOW(), NOW()),
       ('인천광역시 남동구 만수동', '111', '471', '2', '29.52', '202501', '50,000,000', '다세대', NOW(), NOW());


In [17]:
# INSERT 구문 조립
insert_sql = insert_header + ",\n       ".join(values) + ";"

In [18]:
# 파일 저장
file_name = "V44__insert_transaction_sale_rowhouse_202504.sql"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(insert_sql)

print(f"{file_name} 파일이 생성되었습니다.")

V44__insert_transaction_sale_rowhouse_202504.sql 파일이 생성되었습니다.
